# Toxic Comment Classification — Cross-Dataset Evaluation
**Models implemented from scratch** (no nn.LSTM, nn.GRU, nn.Transformer used).
**Datasets**: Jigsaw (6-label) + HateXplain (3-class) — trained independently.

### Optimizations Applied
1. **Masked pooling** — attention_mask now excludes padding tokens in all models
2. **AMP (Mixed Precision)** — ~2x speedup, lower memory
3. **DataLoader tuning** — num_workers=4, pin_memory=True, persistent_workers
4. **Gradient accumulation** — effective batch of 256
5. **Focal Loss** — better for class imbalance
6. **EnhancedRCNN** — deeper fusion layers
7. **Pre-LN Transformer + CLS token** — more stable, 4 layers
8. **Label smoothing** — better calibration
9. **OneCycleLR scheduler** — faster convergence

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q transformers pandas scikit-learn tqdm

import os
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import DistilBertTokenizer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ============================================================
# HYPERPARAMETERS
# ============================================================
LABEL_COLS_JIGSAW = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
LABEL_COLS_HATEXPLAIN = ['normal', 'offensive', 'hatespeech']

EMBEDDING_DIM = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.3
MAX_LEN = 128
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10
GRAD_ACCUM = 4
NUM_WORKERS = 4

MODEL_CONFIGS = {
    'lstm':           {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'bilstm':         {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'attentionlstm':  {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'attentionbilstm':{'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'gru':            {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'rcnn':           {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'enhancedrcnn':   {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
    'transformer':    {'model_class': None, 'hidden_size': HIDDEN_SIZE, 'num_layers': 4, 'dropout': DROPOUT},
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ============================================================
# TOKENIZER
# ============================================================
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
VOCAB_SIZE = len(tokenizer)
print(f'Vocab size: {VOCAB_SIZE}')

# ============================================================
# UTILITY FUNCTIONS
# ============================================================

In [ ]:
# ============================================================
# MASKED POOLING (Phase 1.1 — Bug Fix)
# attention_mask was passed but ignored — padding tokens polluted pooled representations
# ============================================================

def masked_mean_pool(tensor, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).float()
    mask_sum = mask_expanded.sum(dim=1).clamp(min=1)
    sum_embeds = (tensor * mask_expanded).sum(dim=1)
    return sum_embeds / mask_sum

def masked_max_pool(tensor, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).float()
    tensor_masked = tensor.masked_fill(mask_expanded == 0, float('-inf'))
    out, _ = tensor_masked.max(dim=1)
    return out

# ============================================================
# FOCAL LOSS (Phase 2.2)
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, num_classes=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        probs = torch.softmax(logits, dim=1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma
        return (focal_weight * ce).mean()

def get_loss_function(loss_type, num_classes, pos_weight=None):
    if loss_type == 'focal':
        return FocalLoss(num_classes=num_classes)
    elif loss_type == 'crossentropy':
        return nn.CrossEntropyLoss()
    else:
        return nn.CrossEntropyLoss()

# ============================================================
# METRICS (supports both multi-label and multi-class)
# ============================================================

def compute_metrics_multi_label(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    metrics = {}
    try:
        metrics['auc'] = roc_auc_score(labels, probs, average='macro')
        metrics['auc_per_class'] = roc_auc_score(labels, probs, average=None)
    except:
        metrics['auc'] = 0.0
        metrics['auc_per_class'] = np.zeros(labels.shape[1])
    metrics['f1_macro'] = f1_score(labels, preds, average='macro', zero_division=0)
    metrics['f1_micro'] = f1_score(labels, preds, average='micro', zero_division=0)
    metrics['precision'] = precision_score(labels, preds, average='macro', zero_division=0)
    metrics['recall'] = recall_score(labels, preds, average='macro', zero_division=0)
    return metrics

def compute_metrics_multi_class(labels, probs):
    preds = probs.argmax(axis=1)
    metrics = {}
    try:
        metrics['auc'] = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
        metrics['auc_per_class'] = roc_auc_score(labels, probs, multi_class='ovr', average=None)
    except:
        metrics['auc'] = 0.0
        metrics['auc_per_class'] = np.zeros(probs.shape[1])
    metrics['f1_macro'] = f1_score(labels, preds, average='macro', zero_division=0)
    metrics['f1_micro'] = f1_score(labels, preds, average='micro', zero_division=0)
    metrics['precision'] = precision_score(labels, preds, average='macro', zero_division=0)
    metrics['recall'] = recall_score(labels, preds, average='macro', zero_division=0)
    metrics['accuracy'] = (preds == labels).mean()
    return metrics

# ============================================================
# FROM-SCRATCH RNN CELLS & ATTENTION
# ============================================================

In [ ]:
# ============================================================
# CUSTOM LSTM CELL (from scratch)
# ============================================================

class OwnLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(OwnLSTMCell, self).__init__()
        self.hidden_size = hidden_size
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, hidden_state):
        h, c = hidden_state
        combined = torch.cat([h, x], dim=1)
        i = torch.sigmoid(self.W_i(combined))
        f = torch.sigmoid(self.W_f(combined))
        c_tilde = torch.tanh(self.W_c(combined))
        o = torch.sigmoid(self.W_o(combined))
        c = f * c + i * c_tilde
        h = o * torch.tanh(c)
        return h, c

# ============================================================
# CUSTOM GRU CELL (from scratch)
# ============================================================

class OwnGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(OwnGRUCell, self).__init__()
        self.hidden_size = hidden_size
        self.W_z = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_r = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_h = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h):
        combined = torch.cat([h, x], dim=1)
        z = torch.sigmoid(self.W_z(combined))
        r = torch.sigmoid(self.W_r(combined))
        h_combined = torch.cat([r * h, x], dim=1)
        h_tilde = torch.tanh(self.W_h(h_combined))
        h = (1 - z) * h + z * h_tilde
        return h

# ============================================================
# SELF-ATTENTION (from scratch — additive attention)
# ============================================================

class SelfAttention(nn.Module):
    def __init__(self, hidden_size):
        super(SelfAttention, self).__init__()
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states, mask=None):
        energy = torch.tanh(self.projection(hidden_states))
        weights = self.v(energy)
        if mask is not None:
            mask = mask.unsqueeze(-1).float()
            weights = weights.masked_fill(mask == 0, -1e9)
        weights = F.softmax(weights, dim=1)
        context = torch.sum(weights * hidden_states, dim=1)
        return context, weights

# ============================================================
# TRANSFORMER COMPONENTS (from scratch)
# ============================================================

In [ ]:
# ============================================================
# POSITIONAL ENCODING
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ============================================================
# MULTI-HEAD SELF-ATTENTION (from scratch)
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        Q = self.W_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(context)

# ============================================================
# PRE-LN TRANSFORMER ENCODER BLOCK (Phase 2.3)
# Changed from Post-LN to Pre-LN — more stable training
# ============================================================

class PreLNTransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, feedforward_dim, dropout):
        super(PreLNTransformerEncoderBlock, self).__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, feedforward_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(feedforward_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x_norm = self.norm1(x)
        attn_out = self.attention(x_norm, x_norm, x_norm, mask=mask)
        x = x + self.dropout(attn_out)
        x_norm2 = self.norm2(x)
        ff_out = self.ff(x_norm2)
        x = x + self.dropout(ff_out)
        return x

# ============================================================
# ALL MODEL CLASSES
# All models are dataset-agnostic — num_classes passed at init
# ============================================================

In [ ]:
# ============================================================
# OWN LSTM
# ============================================================

class OwnLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(OwnLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size, hidden_size)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        out = self.embedding(x)
        for layer_idx in range(self.num_layers):
            h = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c = torch.zeros(batch_size, self.hidden_size).to(x.device)
            layer_outputs = []
            for t in range(seq_len):
                h, c = self.layers[layer_idx](out[:, t, :], (h, c))
                layer_outputs.append(h)
            out = torch.stack(layer_outputs, dim=1)
            if layer_idx < self.num_layers - 1:
                out = self.dropout(out)
        pooled = masked_mean_pool(out, attention_mask) if attention_mask is not None else torch.mean(out, dim=1)
        pooled = self.dropout(pooled)
        return self.fc(pooled)

# ============================================================
# OWN BiLSTM
# ============================================================

class OwnBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(OwnBiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.bwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        out = self.embedding(x)
        for layer_idx in range(self.num_layers):
            h_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            h_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            fwd_outputs = []
            bwd_outputs = [None] * seq_len
            for t in range(seq_len):
                h_fwd, c_fwd = self.fwd_layers[layer_idx](out[:, t, :], (h_fwd, c_fwd))
                fwd_outputs.append(h_fwd)
            for t in range(seq_len - 1, -1, -1):
                h_bwd, c_bwd = self.bwd_layers[layer_idx](out[:, t, :], (h_bwd, c_bwd))
                bwd_outputs[t] = h_bwd
            fwd_outputs = torch.stack(fwd_outputs, dim=1)
            bwd_outputs = torch.stack(bwd_outputs, dim=1)
            out = torch.cat((fwd_outputs, bwd_outputs), dim=2)
            if layer_idx < self.num_layers - 1:
                out = self.dropout(out)
        pooled = masked_mean_pool(out, attention_mask) if attention_mask is not None else torch.mean(out, dim=1)
        pooled = self.dropout(pooled)
        return self.fc(pooled)

# ============================================================
# ATTENTION LSTM
# ============================================================

class AttentionLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(AttentionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size, hidden_size)
            for i in range(num_layers)
        ])
        self.attention = SelfAttention(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        out = self.embedding(x)
        for layer_idx in range(self.num_layers):
            h = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c = torch.zeros(batch_size, self.hidden_size).to(x.device)
            layer_outputs = []
            for t in range(seq_len):
                h, c = self.layers[layer_idx](out[:, t, :], (h, c))
                layer_outputs.append(h)
            out = torch.stack(layer_outputs, dim=1)
            if layer_idx < self.num_layers - 1:
                out = self.dropout(out)
        context, _ = self.attention(out, mask=attention_mask)
        context = self.dropout(context)
        return self.fc(context)

# ============================================================
# ATTENTION BiLSTM
# ============================================================

class AttentionBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(AttentionBiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.bwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.attention = SelfAttention(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        out = self.embedding(x)
        for layer_idx in range(self.num_layers):
            h_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            h_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            fwd_outputs = []
            bwd_outputs = [None] * seq_len
            for t in range(seq_len):
                h_fwd, c_fwd = self.fwd_layers[layer_idx](out[:, t, :], (h_fwd, c_fwd))
                fwd_outputs.append(h_fwd)
            for t in range(seq_len - 1, -1, -1):
                h_bwd, c_bwd = self.bwd_layers[layer_idx](out[:, t, :], (h_bwd, c_bwd))
                bwd_outputs[t] = h_bwd
            fwd_outputs = torch.stack(fwd_outputs, dim=1)
            bwd_outputs = torch.stack(bwd_outputs, dim=1)
            out = torch.cat((fwd_outputs, bwd_outputs), dim=2)
            if layer_idx < self.num_layers - 1:
                out = self.dropout(out)
        context, _ = self.attention(out, mask=attention_mask)
        context = self.dropout(context)
        return self.fc(context)

# ============================================================
# OWN GRU
# ============================================================

class OwnGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(OwnGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.layers = nn.ModuleList([
            OwnGRUCell(embedding_dim if i == 0 else hidden_size, hidden_size)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        out = self.embedding(x)
        for layer_idx in range(self.num_layers):
            h = torch.zeros(batch_size, self.hidden_size).to(x.device)
            layer_outputs = []
            for t in range(seq_len):
                h = self.layers[layer_idx](out[:, t, :], h)
                layer_outputs.append(h)
            out = torch.stack(layer_outputs, dim=1)
            if layer_idx < self.num_layers - 1:
                out = self.dropout(out)
        pooled = masked_mean_pool(out, attention_mask) if attention_mask is not None else torch.mean(out, dim=1)
        pooled = self.dropout(pooled)
        return self.fc(pooled)

# ============================================================
# OWN RCNN
# ============================================================

class OwnRCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(OwnRCNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.bwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.fusion = nn.Linear(hidden_size * 2 + embedding_dim, hidden_size * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        embeds = self.embedding(x)
        rnn_out = embeds
        for layer_idx in range(self.num_layers):
            h_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            h_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            fwd_outputs = []
            bwd_outputs = [None] * seq_len
            for t in range(seq_len):
                h_fwd, c_fwd = self.fwd_layers[layer_idx](rnn_out[:, t, :], (h_fwd, c_fwd))
                fwd_outputs.append(h_fwd)
            for t in range(seq_len - 1, -1, -1):
                h_bwd, c_bwd = self.bwd_layers[layer_idx](rnn_out[:, t, :], (h_bwd, c_bwd))
                bwd_outputs[t] = h_bwd
            fwd_outputs = torch.stack(fwd_outputs, dim=1)
            bwd_outputs = torch.stack(bwd_outputs, dim=1)
            rnn_out = torch.cat((fwd_outputs, bwd_outputs), dim=2)
            if layer_idx < self.num_layers - 1:
                rnn_out = self.dropout(rnn_out)
        combined = torch.cat((rnn_out[:, :, :self.hidden_size], embeds, rnn_out[:, :, self.hidden_size:]), dim=2)
        latent = torch.tanh(self.fusion(combined))
        pooled = masked_max_pool(latent, attention_mask) if attention_mask is not None else latent.max(dim=1)[0]
        pooled = self.dropout(pooled)
        return self.fc(pooled)

# ============================================================
# ENHANCED RCNN (Phase 2.1 — deeper fusion layers)
# ============================================================

class EnhancedRCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout):
        super(EnhancedRCNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.bwd_layers = nn.ModuleList([
            OwnLSTMCell(embedding_dim if i == 0 else hidden_size * 2, hidden_size)
            for i in range(num_layers)
        ])
        self.fusion = nn.Linear(hidden_size * 2 + embedding_dim, hidden_size * 2)
        self.fc_hidden = nn.Linear(hidden_size * 2, hidden_size)
        self.fc_final = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.size()
        embeds = self.embedding(x)
        rnn_out = embeds
        for layer_idx in range(self.num_layers):
            h_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_fwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            h_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_bwd = torch.zeros(batch_size, self.hidden_size).to(x.device)
            fwd_outputs = []
            bwd_outputs = [None] * seq_len
            for t in range(seq_len):
                h_fwd, c_fwd = self.fwd_layers[layer_idx](rnn_out[:, t, :], (h_fwd, c_fwd))
                fwd_outputs.append(h_fwd)
            for t in range(seq_len - 1, -1, -1):
                h_bwd, c_bwd = self.bwd_layers[layer_idx](rnn_out[:, t, :], (h_bwd, c_bwd))
                bwd_outputs[t] = h_bwd
            fwd_outputs = torch.stack(fwd_outputs, dim=1)
            bwd_outputs = torch.stack(bwd_outputs, dim=1)
            rnn_out = torch.cat((fwd_outputs, bwd_outputs), dim=2)
            if layer_idx < self.num_layers - 1:
                rnn_out = self.dropout(rnn_out)
        combined = torch.cat((rnn_out[:, :, :self.hidden_size], embeds, rnn_out[:, :, self.hidden_size:]), dim=2)
        latent = torch.tanh(self.fusion(combined))
        pooled = masked_max_pool(latent, attention_mask) if attention_mask is not None else latent.max(dim=1)[0]
        out = torch.relu(self.fc_hidden(pooled))
        out = self.dropout(out)
        return self.fc_final(out)

# ============================================================
# OWN TRANSFORMER (Pre-LN + CLS token, 4 layers)
# ============================================================

class OwnTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout, num_heads=8):
        super(OwnTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding = PositionalEncoding(embedding_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        self.encoder_layers = nn.ModuleList([
            PreLNTransformerEncoderBlock(
                d_model=embedding_dim,
                num_heads=num_heads,
                feedforward_dim=hidden_size * 2,
                dropout=dropout
            ) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, attention_mask=None):
        batch_size = x.size(0)
        out = self.embedding(x)
        out = self.pos_encoding(out)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        out = torch.cat([cls_tokens, out], dim=1)
        for layer in self.encoder_layers:
            out = layer(out, mask=attention_mask)
        cls_output = out[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.fc(cls_output)

# ============================================================
# DATASET CLASSES (per-dataset, independent label schemes)
# ============================================================

In [ ]:
# ============================================================
# JIGSAW DATASET — 6-label multi-label classification
# ============================================================

class JigsawDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

# ============================================================
# HATEXPLAIN DATASET — 3-class single-label classification
# comment + label (normal/offensive/hatespeech)
# Label encoded as integer: normal=0, offensive=1, hatespeech=2
# ============================================================

LABEL_MAP_HATEXPLAIN = {'normal': 0, 'offensive': 1, 'hatespeech': 2}
LABEL_NAMES_HATEXPLAIN = ['normal', 'offensive', 'hatespeech']

class HateXplainDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ============================================================
# TRAINING FUNCTIONS (AMP + gradient accumulation)
# Supports both multi-label (Jigsaw) and multi-class (HateXplain)
# ============================================================

In [ ]:
# ============================================================
# BUILD MODEL (dataset-agnostic — num_classes from config)
# ============================================================

def build_model(model_name, vocab_size, embedding_dim, num_classes):
    config = MODEL_CONFIGS[model_name]
    return config['model_class'](
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        num_classes=num_classes,
        dropout=config['dropout']
    )

# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_epoch(model, dataloader, criterion, optimizer, device, scaler=None, grad_accum=1, scheduler=None):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    pbar = tqdm(dataloader, desc='Training')
    for batch_idx, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels) / grad_accum
            scaler.scale(loss).backward()
            if (batch_idx + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if scheduler is not None:
                    scheduler.step()
        else:
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / grad_accum
            loss.backward()
            if (batch_idx + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                if scheduler is not None:
                    scheduler.step()
        
        total_loss += loss.item() * grad_accum
        pbar.set_postfix({'loss': f'{loss.item() * grad_accum:.4f}'})
    
    return total_loss / len(dataloader)

# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, dataloader, criterion, device, is_multi_label=False):
    model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            if is_multi_label:
                probs = torch.sigmoid(logits).cpu().numpy()
            else:
                probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs)
    
    all_labels = np.vstack(all_labels) if is_multi_label else np.concatenate(all_labels)
    all_probs = np.vstack(all_probs) if is_multi_label else np.vstack(all_probs)
    
    if is_multi_label:
        metrics = compute_metrics_multi_label(all_labels, all_probs)
    else:
        metrics = compute_metrics_multi_class(all_labels, all_probs)
    metrics['loss'] = total_loss / len(dataloader)
    
    return metrics, all_labels, all_probs

# ============================================================
# MAIN TRAINING LOOP (per-dataset)
# ============================================================

In [ ]:
def train_model(
    model_name, train_dataset, val_dataset,
    num_classes, is_multi_label,
    device, epochs=10, batch_size=64, lr=1e-3,
    loss_type='crossentropy', use_amp=True, num_workers=4,
    grad_accum=1, patience=3
):
    model = build_model(model_name, VOCAB_SIZE, EMBEDDING_DIM, num_classes).to(device)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False,
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )
    
    criterion = get_loss_function(loss_type, num_classes)
    val_criterion = get_loss_function(loss_type, num_classes)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = (len(train_loader) * epochs // grad_accum)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1
    )
    
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    best_auc = 0
    best_metrics = None
    patience_counter = 0
    
    for epoch in range(epochs):
        print(f'\n=== Epoch {epoch+1}/{epochs} ===')
        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, device,
            scaler, grad_accum, scheduler
        )
        val_metrics, _, _ = evaluate(
            model, val_loader, val_criterion, device, is_multi_label
        )
        
        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_metrics["loss"]:.4f} | '
              f'Val AUC: {val_metrics["auc"]:.4f} | Val F1 Macro: {val_metrics["f1_macro"]:.4f}')
        
        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            best_metrics = val_metrics
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    return best_metrics, best_auc

def run_all_models(
    dataset_name, train_dataset, val_dataset,
    num_classes, is_multi_label,
    epochs=10, batch_size=64, lr=1e-3, loss_type='crossentropy'
):
    set_seed(42)
    print(f'\n{"="*60}')
    print(f'Training all models on: {dataset_name} (num_classes={num_classes})')
    print(f'{"="*60}')
    print(f'Train size: {len(train_dataset)}, Val size: {len(val_dataset)}')
    
    results = {}
    for model_name in MODEL_CONFIGS:
        print(f'\n{"="*50}')
        print(f'Training {model_name} on {dataset_name}')
        print(f'{"="*50}')
        metrics, best_auc = train_model(
            model_name, train_dataset, val_dataset,
            num_classes, is_multi_label,
            DEVICE,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            loss_type=loss_type,
            use_amp=True,
            num_workers=NUM_WORKERS,
            grad_accum=GRAD_ACCUM
        )
        results[model_name] = {
            'auc': best_auc,
            'f1_macro': metrics['f1_macro'],
            'f1_micro': metrics['f1_micro'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
        }
        print(f'{model_name} Best Val AUC: {best_auc:.4f}')
    
    print('\n' + '='*60)
    print(f'Results for {dataset_name}')
    print('='*60)
    print(f'{"Model":<20} | {"AUC":>8} | {"F1 Macro":>9} | {"F1 Micro":>9} | {"Prec":>7} | {"Rec":>7}')
    print('-'*70)
    for model_name, m in sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True):
        print(f'{model_name:<20} | {m["auc"]:>8.4f} | {m["f1_macro"]:>9.4f} | {m["f1_micro"]:>9.4f} | {m["precision"]:>7.4f} | {m["recall"]:>7.4f}')
    
    return results

# ============================================================
# EXPERIMENT A: JIGSAW DATASET (6-label multi-label)
# Path: /kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge
# ============================================================

In [ ]:
# ============================================================
# LOAD JIGSAW DATA
# ============================================================

DATA_DIR_JIGSAW = "/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge"
jigsaw_df = pd.read_csv(os.path.join(DATA_DIR_JIGSAW, 'train.csv'))
jigsaw_texts = jigsaw_df['comment_text'].fillna('').tolist()
jigsaw_labels = jigsaw_df[LABEL_COLS_JIGSAW].values.tolist()

print(f'Jigsaw dataset: {len(jigsaw_texts)} samples')
print(f'Label distribution: {dict(zip(LABEL_COLS_JIGSAW, jigsaw_df[LABEL_COLS_JIGSAW].mean().round(4)))}')

# Split
jigsaw_full = JigsawDataset(jigsaw_texts, jigsaw_labels, tokenizer, MAX_LEN)
jigsaw_train_size = int(len(jigsaw_full) * 0.8)
jigsaw_val_size = len(jigsaw_full) - jigsaw_train_size
jigsaw_train, jigsaw_val = random_split(
    jigsaw_full, [jigsaw_train_size, jigsaw_val_size],
    generator=torch.Generator().manual_seed(42)
)

# Run all models on Jigsaw (multi-label, 6 classes)
jigsaw_results = run_all_models(
    dataset_name='Jigsaw',
    train_dataset=jigsaw_train,
    val_dataset=jigsaw_val,
    num_classes=6,
    is_multi_label=True,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    loss_type='crossentropy'
)

# ============================================================
# EXPERIMENT B: HATEXPLAIN DATASET (3-class single-label)
# Path: /kaggle/input/sayankr007/cyber-bullying-data-for-multi-label-classification/final_hateXplain.csv
# Labels: normal / offensive / hatespeech
# ============================================================

In [ ]:
# ============================================================
# LOAD HATEXPLAIN DATA
# ============================================================

HATEXPLAIN_PATH = '/kaggle/input/sayankr007/cyber-bullying-data-for-multi-label-classification/final_hateXplain.csv'
hatexplain_df = pd.read_csv(HATEXPLAIN_PATH)

hatexplain_texts = hatexplain_df['comment'].fillna('').tolist()
hatexplain_labels = [LABEL_MAP_HATEXPLAIN[lbl] for lbl in hatexplain_df['label']]

print(f'HateXplain dataset: {len(hatexplain_texts)} samples')
print(f'Label distribution: normal={hatexplain_labels.count(0)}, offensive={hatexplain_labels.count(1)}, hatespeech={hatexplain_labels.count(2)}')

# Split (stratified)
stratify_key = np.array(hatexplain_labels)
train_idx, val_idx = train_test_split(
    np.arange(len(hatexplain_texts)),
    test_size=0.2,
    random_state=42,
    stratify=stratify_key
)

hatexplain_full = HateXplainDataset(hatexplain_texts, hatexplain_labels, tokenizer, MAX_LEN)
hatexplain_train = torch.utils.data.Subset(hatexplain_full, train_idx)
hatexplain_val = torch.utils.data.Subset(hatexplain_full, val_idx)

# Run all models on HateXplain (multi-class, 3 classes)
hatexplain_results = run_all_models(
    dataset_name='HateXplain',
    train_dataset=hatexplain_train,
    val_dataset=hatexplain_val,
    num_classes=3,
    is_multi_label=False,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    loss_type='crossentropy'
)

# ============================================================
# CROSS-DATASET COMPARISON
# Same 8 model architectures, independent training per dataset
# ============================================================

In [ ]:
print('\n' + '='*90)
print('CROSS-DATASET COMPARISON')
print('Same 8 model architectures trained independently on each dataset')
print('='*90)
print(f'{"Model":<20} | {"Jigsaw AUC":>11} | {"Jigsaw F1-M":>11} | {"HateXplain AUC":>13} | {"HateXplain F1-M":>14}')
print('-'*90)
for model_name in MODEL_CONFIGS:
    j = jigsaw_results.get(model_name, {})
    h = hatexplain_results.get(model_name, {})
    jauc = j.get('auc', 0)
    jf1 = j.get('f1_macro', 0)
    hauc = h.get('auc', 0)
    hf1 = h.get('f1_macro', 0)
    print(f'{model_name:<20} | {jauc:>11.4f} | {jf1:>11.4f} | {hauc:>13.4f} | {hf1:>14.4f}')

print('\nNote: Jigsaw uses 6-label multi-label AUC; HateXplain uses 3-class multi-class AUC (ovr, macro)')
print('Same metrics (AUC, F1 Macro) are reported for both — enabling direct comparison.')